# Расследование падения search-to-book conversion

## tl;dr

В synthetic scenario общая conversion снижается с **16,1% до 12,6% (-3,49 п.п.)** после rollout 20 июля 2026. SQL/Pandas decomposition показывает, что главный вклад — **mix shift (-3,59 п.п.)**, а не общий within-segment спад. Одновременно локальный путь Android × Moscow × paid_search для 5.14.0 падает примерно на **8,13 п.п.**; это operationally важная гипотеза, но не causal claim без holdback.

> Все данные детерминированно сгенерированы для учебного кейса и не относятся к реальной компании.

## Context & Methods

**Вопрос.** Что объясняет падение session-level search-to-book conversion после release date?

### Key Assumptions

- Единица анализа — search-сессия; booking должен следовать за search в той же session.
- Окна `before` (29 июня—19 июля) и `after` (20 июля—9 августа) равной длины; timestamps synthetic и в UTC.
- `session_quality`, channel, platform и city известны на момент search; booking формируется только из raw events.
- Before/after и версии приложения — наблюдательные признаки. Они полезны для диагностики, но не идентифицируют causal effect.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from rca.analysis import (
    build_funnel_with_pandas,
    data_quality_checks,
    load_raw_data,
    run_analysis,
    run_sql_models,
)
from rca.generator import generate_synthetic_data

DATA_DIR = ROOT / 'data' / 'generated'
REPORTS_DIR = ROOT / 'reports'
if not (DATA_DIR / 'events.csv').exists():
    generate_synthetic_data(DATA_DIR)


## Data

Загружаем raw events, attributes и release calendar. В следующей ячейке pandas фильтрует только нужные events, строит `groupby` rollup и делает проверяемый `merge` с измерениями сессии.

In [2]:
events, attributes, releases = load_raw_data(DATA_DIR)
funnel_pandas = build_funnel_with_pandas(events, attributes)

print(f'Raw events: {len(events):,}; session attributes: {len(attributes):,}')
display(events.head(4))
display(
    funnel_pandas.groupby('period', as_index=False)
    .agg(search_sessions=('session_id', 'size'), bookings=('has_booking', 'sum'))
    .assign(conversion=lambda x: x.bookings / x.search_sessions)
)


Raw events: 124,066; session attributes: 108,626


,session_id,user_id,event_id,event_name,event_ts,ingested_at
0,s_0000896,u_0000896,e_search_0000896,search_results_viewed,2026-06-29 08:00:13,2026-06-29 15:00:13
1,s_0002213,u_0002213,e_search_0002213,search_results_viewed,2026-06-29 08:00:14,2026-06-29 10:00:14
2,s_0000742,u_0000742,e_search_0000742,search_results_viewed,2026-06-29 08:00:21,2026-06-29 13:00:21
3,s_0000217,u_0000217,e_search_0000217,search_results_viewed,2026-06-29 08:00:53,2026-06-29 23:00:53


,period,search_sessions,bookings,conversion
0,after,58472,7370,0.126043
1,before,50154,8070,0.160904


## Results

Здесь SQL CTE создают session mart и decomposition, а независимый pandas path служит cross-check. Отдельно запускаются freshness, uniqueness, event-order и join-coverage проверки.

In [3]:
mart_sql, contribution, sql_quality = run_sql_models(DATA_DIR)
quality = data_quality_checks(events, attributes, releases, mart_sql, sql_quality)
result = run_analysis(DATA_DIR, REPORTS_DIR)
metrics = result['metrics']

print('All data-quality checks passed:', quality['all_passed'])
display(pd.DataFrame(quality['checks']).T[['passed', 'value', 'rule']])
display(contribution[['acquisition_channel', 'session_quality', 'mix_effect', 'rate_effect', 'total_effect']])


All data-quality checks passed: True


,passed,value,rule
event_id_uniqueness,True,0,0 duplicate event_id values
session_key_uniqueness,True,0,0 duplicate session_id values in the session d...
required_attributes,True,0,0 nulls in session-grain dimensions
event_order,True,0,booking_created cannot precede search_results_...
join_coverage,True,0,every session attribute row maps to exactly on...
date_coverage,True,0,no missing daily partitions in the analysis wi...
freshness,True,20.57,latest ingestion is within 24 hours of the as-...
sql_event_grain,True,"{'duplicate_event_ids': 0, 'sessions_without_e...",SQL event-grain checks are all zero


,acquisition_channel,session_quality,mix_effect,rate_effect,total_effect
0,organic,high,-0.017577,0.000645,-0.016932
1,organic,standard,-0.012358,0.000978,-0.011379
2,paid_search,standard,-0.008429,-0.001348,-0.009777
3,paid_search,high,-0.006457,-0.000897,-0.007354
4,paid_social,standard,0.006203,-0.000249,0.005954
5,paid_social,low,0.006060,0.002744,0.008804
6,paid_social,high,0.002050,-0.000491,0.001559
7,referral,high,-0.001756,0.000069,-0.001687
8,organic,low,-0.001581,-0.000181,-0.001761
9,referral,standard,-0.001400,0.000405,-0.000995


In [4]:
before = metrics['period_totals']['before']
after = metrics['period_totals']['after']
local = metrics['local_release_slice']
print(f"Global: {before['conversion']:.2%} → {after['conversion']:.2%}; Δ {metrics['global_delta']:.2%}")
print(f"Mix effect: {metrics['decomposition']['mix_effect']:.2%}; within-cell rate effect: {metrics['decomposition']['rate_effect']:.2%}")
print(f"Local Android/Moscow/paid-search 5.14.0: {local['before_rate']:.2%} → {local['after_rate']:.2%}; 95% CI for Δ: [{local['ci_low']:.2%}, {local['ci_high']:.2%}]")
display(pd.DataFrame(local['by_quality']))
# PNG files are saved and versioned separately. Linking avoids platform-dependent
# binary MIME payloads in this executed notebook while retaining the useful charts.
display(HTML(
    '<p><strong>Saved figures:</strong> '
    '<a href="../reports/figures/mix_decomposition.png">mix decomposition</a> · '
    '<a href="../reports/figures/local_regression.png">local regression</a></p>'
))


Global: 16.09% → 12.60%; Δ -3.49%
Mix effect: -3.59%; within-cell rate effect: 0.10%
Local Android/Moscow/paid-search 5.14.0: 16.22% → 8.09%; 95% CI for Δ: [-9.99%, -6.28%]


,session_quality,before_rate,after_rate,difference,ci_low,ci_high,z_score,p_value
0,high,0.258571,0.177384,-0.081188,-0.129094,-0.033282,-3.211077,1.322387e-03
1,standard,0.164158,0.082294,-0.081863,-0.109059,-0.054668,-5.418300,6.016838e-08
2,low,0.057949,0.006645,-0.051305,-0.070112,-0.032497,-5.066180,4.058772e-07


## Takeaways

1. Не делаем вывод «релиз испортил общую conversion» только из aggregate before/after: composition traffic меняет взвешенную среднюю.
2. Локальный Android-срез достаточно велик и стабилен по session quality, чтобы стать P0-проверкой: feature flag/rollback, telemetry, availability и latency.
3. Для подтверждения эффекта после фикса нужен заранее зарегистрированный randomised holdback; p-value в diagnostic slice не заменяет эксперимент и не снимает риск multiple testing.

Reader-facing отчёт, SQL, data dictionary, figures и final validation находятся в корне репозитория.